# Flipkart Reviews Sentiment Analysis

This notebook demonstrates the complete sentiment analysis pipeline:
1. Load Dataset
2. Exploratory Data Analysis (EDA)
3. Sentiment Creation
4. Data Cleaning
5. Visualizations
6. Feature Engineering
7. Model Training
8. Model Evaluation
9. Model Comparison
10. Analytical Results
11. Sample Prediction

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import re
import os
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from wordcloud import WordCloud

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, classification_report, confusion_matrix)

nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)

sns.set_style('whitegrid')
print('All libraries imported successfully!')

## 2. Load Dataset

In [ ]:
df = pd.read_csv(r'C:\Users\Prem Jha\python_tp\Flipkart-Sentiment-Analysis\data\flipkart_data.csv')
print('Dataset loaded successfully!')

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Dataset shape
print('Dataset Shape:', df.shape)
print('Rows   :', df.shape[0])
print('Columns:', df.shape[1])

In [ ]:
# First five rows
df.head()

In [ ]:
# Missing values
print('Missing Values:')
print(df.isnull().sum())

In [ ]:
# Rating distribution
print('Rating Distribution:')
print(df['rating'].value_counts().sort_index())

plt.figure(figsize=(8, 4))
sns.countplot(x='rating', data=df, palette='viridis')
plt.title('Rating Distribution')
plt.xlabel('Rating')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

## 4. Sentiment Creation

- Rating 4 or 5 -> Positive
- Rating 3 -> Neutral
- Rating 1 or 2 -> Negative

In [ ]:
def get_sentiment(rating):
    if rating >= 4:
        return 'Positive'
    elif rating == 3:
        return 'Neutral'
    else:
        return 'Negative'

df['sentiment'] = df['rating'].apply(get_sentiment)

# Sentiment distribution
print('Sentiment Distribution:')
print(df['sentiment'].value_counts())

df[['rating', 'sentiment']].head(10)

## 5. Data Cleaning

Steps:
- Lowercase
- Remove URLs
- Remove punctuation
- Remove numbers
- Remove extra spaces
- Tokenize
- Remove stopwords

In [ ]:
stop_words = set(stopwords.words('english'))

def clean_text(text):
    # Lowercase
    text = str(text).lower()
    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    # Remove punctuation and numbers
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()
    # Tokenize
    words = word_tokenize(text)
    # Remove stopwords
    words = [word for word in words if word not in stop_words]
    return ' '.join(words)

df['clean_review'] = df['review'].apply(clean_text)
print('Reviews cleaned successfully!')

# Show before and after
df[['review', 'clean_review']].head()

## 6. Visualizations

In [ ]:
# Sentiment distribution bar chart
plt.figure(figsize=(8, 5))
sns.countplot(x='sentiment', data=df, palette='viridis',
              order=['Positive', 'Neutral', 'Negative'])
plt.title('Sentiment Distribution', fontsize=14)
plt.xlabel('Sentiment')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
# Positive word cloud
pos_text = ' '.join(df[df['sentiment'] == 'Positive']['clean_review'].astype(str))
pos_wc = WordCloud(width=800, height=400, background_color='white',
                   colormap='viridis', max_words=100).generate(pos_text)

plt.figure(figsize=(10, 5))
plt.imshow(pos_wc, interpolation='bilinear')
plt.axis('off')
plt.title('Positive Reviews - Word Cloud', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Negative word cloud
neg_text = ' '.join(df[df['sentiment'] == 'Negative']['clean_review'].astype(str))
neg_wc = WordCloud(width=800, height=400, background_color='white',
                   colormap='magma', max_words=100).generate(neg_text)

plt.figure(figsize=(10, 5))
plt.imshow(neg_wc, interpolation='bilinear')
plt.axis('off')
plt.title('Negative Reviews - Word Cloud', fontsize=14)
plt.tight_layout()
plt.show()

## 7. Feature Engineering (TF-IDF)

In [ ]:
tfidf = TfidfVectorizer(max_features=5000)

X = tfidf.fit_transform(df['clean_review'])
y = df['sentiment']

print('TF-IDF Shape:', X.shape)

## 8. Train-Test Split (80% Training, 20% Testing)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print('Training Set:', X_train.shape[0], 'samples')
print('Testing Set :', X_test.shape[0], 'samples')

## 9. Model Training

In [ ]:
# Model 1: Naive Bayes
nb = MultinomialNB()
nb.fit(X_train, y_train)
pred_nb = nb.predict(X_test)
print('Naive Bayes trained!')

In [ ]:
# Model 2: Logistic Regression
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)
pred_lr = lr.predict(X_test)
print('Logistic Regression trained!')

## 10. Model Evaluation

In [ ]:
# Naive Bayes Evaluation
print('===== Naive Bayes Evaluation =====')
print(f'Accuracy  : {accuracy_score(y_test, pred_nb):.4f}')
print(f'Precision : {precision_score(y_test, pred_nb, average="weighted", zero_division=0):.4f}')
print(f'Recall    : {recall_score(y_test, pred_nb, average="weighted", zero_division=0):.4f}')
print(f'F1 Score  : {f1_score(y_test, pred_nb, average="weighted", zero_division=0):.4f}')
print('\nClassification Report:')
print(classification_report(y_test, pred_nb, zero_division=0))

In [ ]:
# Naive Bayes Confusion Matrix
cm_nb = confusion_matrix(y_test, pred_nb, labels=['Positive', 'Neutral', 'Negative'])
plt.figure(figsize=(7, 5))
sns.heatmap(cm_nb, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Positive', 'Neutral', 'Negative'],
            yticklabels=['Positive', 'Neutral', 'Negative'])
plt.title('Confusion Matrix - Naive Bayes', fontsize=14)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

In [ ]:
# Logistic Regression Evaluation
print('===== Logistic Regression Evaluation =====')
print(f'Accuracy  : {accuracy_score(y_test, pred_lr):.4f}')
print(f'Precision : {precision_score(y_test, pred_lr, average="weighted", zero_division=0):.4f}')
print(f'Recall    : {recall_score(y_test, pred_lr, average="weighted", zero_division=0):.4f}')
print(f'F1 Score  : {f1_score(y_test, pred_lr, average="weighted", zero_division=0):.4f}')
print('\nClassification Report:')
print(classification_report(y_test, pred_lr, zero_division=0))

In [ ]:
# Logistic Regression Confusion Matrix
cm_lr = confusion_matrix(y_test, pred_lr, labels=['Positive', 'Neutral', 'Negative'])
plt.figure(figsize=(7, 5))
sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Positive', 'Neutral', 'Negative'],
            yticklabels=['Positive', 'Neutral', 'Negative'])
plt.title('Confusion Matrix - Logistic Regression', fontsize=14)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

## 11. Model Comparison

In [ ]:
comparison = pd.DataFrame({
    'Model': ['Naive Bayes', 'Logistic Regression'],
    'Accuracy': [
        accuracy_score(y_test, pred_nb),
        accuracy_score(y_test, pred_lr)
    ],
    'Precision': [
        precision_score(y_test, pred_nb, average='weighted', zero_division=0),
        precision_score(y_test, pred_lr, average='weighted', zero_division=0)
    ],
    'Recall': [
        recall_score(y_test, pred_nb, average='weighted', zero_division=0),
        recall_score(y_test, pred_lr, average='weighted', zero_division=0)
    ],
    'F1 Score': [
        f1_score(y_test, pred_nb, average='weighted', zero_division=0),
        f1_score(y_test, pred_lr, average='weighted', zero_division=0)
    ]
})

print('Model Comparison:')
print(comparison.to_string(index=False))

# Highlight best model
best = comparison.loc[comparison['Accuracy'].idxmax(), 'Model']
print(f'\nBest Model: {best}')

## 12. Analytical Results

In [ ]:
# Sentiment percentages
total = len(df)
pos_count = len(df[df['sentiment'] == 'Positive'])
neg_count = len(df[df['sentiment'] == 'Negative'])
neu_count = len(df[df['sentiment'] == 'Neutral'])

print('===== Sentiment Analysis =====')
print(f'Positive Reviews : {pos_count} ({pos_count/total*100:.1f}%)')
print(f'Negative Reviews : {neg_count} ({neg_count/total*100:.1f}%)')
print(f'Neutral Reviews  : {neu_count} ({neu_count/total*100:.1f}%)')

In [ ]:
# Top 20 words in Positive reviews
pos_words = ' '.join(df[df['sentiment'] == 'Positive']['clean_review']).split()
pos_common = Counter(pos_words).most_common(20)

print('Top 20 Words in Positive Reviews:')
for word, count in pos_common:
    print(f'  {word}: {count}')

In [ ]:
# Top 20 words in Negative reviews
neg_words = ' '.join(df[df['sentiment'] == 'Negative']['clean_review']).split()
neg_common = Counter(neg_words).most_common(20)

print('Top 20 Words in Negative Reviews:')
for word, count in neg_common:
    print(f'  {word}: {count}')

## 13. Sample Prediction

In [ ]:
def predict_sentiment(review):
    """Predict sentiment for a given review."""
    cleaned = clean_text(review)
    vector = tfidf.transform([cleaned])
    prediction = lr.predict(vector)
    return prediction[0]

# Test with sample reviews
sample_reviews = [
    'This phone is amazing',
    'Very bad product waste of money',
    'Product is okay nothing special',
    'Excellent quality and fast delivery',
    'Terrible experience never buying again'
]

print('===== Sample Predictions =====\n')
for review in sample_reviews:
    sentiment = predict_sentiment(review)
    print(f'Review: {review}')
    print(f'Predicted Sentiment: {sentiment}\n')